# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [2]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [3]:
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [4]:
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'] 
STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

In [ ]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)
    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        
        if not os.path.exists(genre_path) or not os.path.isdir(genre_path):
            continue
            
        valid_songs = []
        
        for song_folder in os.listdir(genre_path):
            song_path = os.path.join(genre_path, song_folder)     
            is_valid = True
            for stem in STEMS:
                stem_path = os.path.join(song_path, stem)
                
                if not os.path.exists(stem_path):
                    is_valid = False
                    break
                    
                if os.path.getsize(stem_path) < 4096:
                    is_valid = False
                    break
                    
            if is_valid:
                valid_songs.append(song_path)
                
        valid_songs.sort() 
        rng.shuffle(valid_songs)
        
        split_idx = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split_idx]
        val_songs = valid_songs[split_idx:]
        
        # Populate the dictionaries
        def add_to_dict(target, songs, genres):
            for song_path in songs:
                for stem in STEMS:
                    key = stem.replace('.wav', '')
                    target[genres][key].append(os.path.join(song_path, stem))

        add_to_dict(train_dataset, train_songs, genre)
        add_to_dict(val_dataset, val_songs, genre)

    return train_dataset, val_dataset


##### IMPORTANT #####
tr, val = build_dataset(DATA_ROOT, seed=DATA_SEED)
##### IMPORTANT #####

In [ ]:
mb = 1024 * 1024

corrupted_song_folders = 0
stems_below_threshold = 0
stems_above_threshold = 0

LOW_MB = 5.0491 * mb
HIGH_MB = 5.0493 * mb
CORRUPT_LIMIT = 4096

for genre in GENRES:
    genre_dir = os.path.join(DATA_ROOT, genre)
    if not os.path.isdir(genre_dir):
        continue

    for song in os.listdir(genre_dir):
        song_dir = os.path.join(genre_dir, song)
        if not os.path.isdir(song_dir):
            continue

        corrupted_flag = False

        for stem in STEMS:
            stem_file = os.path.join(song_dir, stem)
            if not os.path.isfile(stem_file):
                continue

            size = os.path.getsize(stem_file)

            if size < CORRUPT_LIMIT:
                corrupted_flag = True

            if size < LOW_MB:
                stems_below_threshold += 1
            elif size > HIGH_MB:
                stems_above_threshold += 1

        if corrupted_flag:
            corrupted_song_folders += 1

print("--- Q1 & Q2 Metrics ---")
print(f"Total Corrupted Songs (folders with a file < 4KB): {corrupted_song_folders}")
print(f"Total stems < 5.0491 MB: {stems_below_threshold}")
print(f"Total stems > 5.0493 MB: {stems_above_threshold}")

ans_q1 = corrupted_song_folders + stems_below_threshold
ans_q2 = abs(stems_above_threshold - stems_below_threshold)

print(f"-> Potential Q1 Answer (Corrupted + < 5.0491MB): {ans_q1}")
print(f"-> Potential Q2 Answer (Abs diff > 5.0493MB and < 5.0491MB): {ans_q2}")

reggae_drums_train = len(tr["reggae"]["drums"])
country_vocals_val = len(val["country"]["vocals"])

ans_q3 = abs(reggae_drums_train - country_vocals_val)

print("\n--- Q3 Metrics ---")
print(f"Training Reggae Drums: {reggae_drums_train}")
print(f"Validation Country Vocals: {country_vocals_val}")
print(f"-> Q3 Answer (Absolute Difference): {ans_q3}")


--- Q1 & Q2 Metrics ---
Total Corrupted Songs (folders with a file < 4KB): 0
Total stems < 5.0491 MB: 1256
Total stems > 5.0493 MB: 184
-> Potential Q1 Answer (Corrupted + < 5.0491MB): 1256
-> Potential Q2 Answer (Abs diff > 5.0493MB and < 5.0491MB): 1072

--- Q3 Metrics ---
Training Reggae Drums: 83
Validation Country Vocals: 17
-> Q3 Answer (Absolute Difference): 66


In [ ]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """
    records = []

    total_files = sum(len(file_list) for genre in dataset_dict.values() for file_list in genre.values())

    with tqdm(total=total_files, desc="Processing Audio Files") as pbar:
        for genre, stems in dataset_dict.items():
            for stem, file_list in stems.items():
                for path in file_list:
                    pbar.update(1)

                    y, _ = librosa.load(path, sr=sr)
                    duration = librosa.get_duration(y=y, sr=sr)

                    intervals = librosa.effects.split(
                        y,
                        top_db=top_db,
                        frame_length=N_FFT,
                        hop_length=HOP_LENGTH
                    )

                    max_silence = 0.0
                    silence_locations = []

                    if len(intervals) == 0:
                        max_silence = duration
                        silence_locations.append("FULLY_SILENT")
                    else:
                        intervals_sec = intervals / sr

                        start_gap = intervals_sec[0][0]
                        if start_gap >= threshold_sec:
                            silence_locations.append("START")
                        max_silence = max(max_silence, start_gap)

                        end_gap = duration - intervals_sec[-1][1]
                        if end_gap >= threshold_sec:
                            silence_locations.append("END")
                        max_silence = max(max_silence, end_gap)

                        for i in range(len(intervals_sec) - 1):
                            gap = intervals_sec[i + 1][0] - intervals_sec[i][1]
                            if gap >= threshold_sec and "MIDDLE" not in silence_locations:
                                silence_locations.append("MIDDLE")
                            max_silence = max(max_silence, gap)

                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem,
                            "Duration": round(duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ", ".join(silence_locations),
                            "File_Path": path
                        })

    df = pd.DataFrame(records)
    return df


# --- EXECUTION ---
# Pass your 'tr' (training) dictionary here.
# Ensure 'tr' is defined from your previous build_dataset code.
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

# --- RESULTS ANALYSIS ---

# Create a pivot Table: Count by Genre vs Stem
silence_pivot = df_silence.pivot_table(
    index='Genre', 
    columns='Stem', 
    values='File_Path', 
    aggfunc='count', 
    fill_value=0
)

print("\n--- Silence Distribution (Genre vs. Stem) ---")
print(silence_pivot)

Processing Audio Files: 100%|██████████| 3320/3320 [03:07<00:00, 17.68it/s]


--- Silence Distribution (Genre vs. Stem) ---
Stem       bass  drums  other  vocals
Genre                                
blues        18     25      4      43
classical    70     56      5      69
country      13     13      2      20
disco         6      1      2      19
hiphop       21      2     21       5
jazz         25     22      1      73
metal         6      2      1      42
pop           9      5      2       3
reggae        4      4      8      14
rock         10      7      1      27


In [ ]:

vocals_df = df_silence[df_silence["Stem"] == "vocals"]
jazz_drums_df = df_silence[(df_silence["Stem"] == "drums") & (df_silence["Genre"] == "jazz")]

q4_ans = len(df_silence)
print(f"Q4: Total sound files having silence >= 5 secs: {q4_ans}")

q5_ans = len(vocals_df)
print(f"Q5: Total sound tracks in Vocals with silence >= 5 secs: {q5_ans}")

q6_ans = vocals_df["Max_Silence_Sec"].mean()
print(f"Q6: Average Silence Length in Vocals: {q6_ans:.2f}")

q7_ans = len(jazz_drums_df)
print(f"Q7: Total drums in jazz with silence >= 5 secs: {q7_ans}")

q8_ans = len(jazz_drums_df[jazz_drums_df["Silence_Location"] == "MIDDLE"])
print(f"Q8: Total drums in jazz (silence >= 5s, ONLY middle): {q8_ans}")

q9_ans = len(jazz_drums_df[jazz_drums_df["Max_Silence_Sec"] >= 10.0])
print(f"Q9: Total drums in jazz (silence >= 5s, Max Silence >= 10s): {q9_ans}")


Q4: Total sound files having silence >= 5 secs: 681
Q5: Total sound tracks in Vocals with silence >= 5 secs: 315
Q6: Average Silence Length in Vocals: 12.57
Q7: Total drums in jazz with silence >= 5 secs: 22
Q8: Total drums in jazz (silence >= 5s, ONLY middle): 14
Q9: Total drums in jazz (silence >= 5s, Max Silence >= 10s): 6


In [ ]:
stems_audio = []
try:
    for key in STEM_KEYS:
        file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
        y, _ = librosa.load(file_path, sr=SR, duration=DURATION)
        stems_audio.append(y)

    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

Audio loaded successfully.


In [ ]:

stems_stack = np.stack(stems_audio)

mix_signal = stems_stack.sum(axis=0)

rms_value = np.sqrt(np.mean(mix_signal ** 2))

peak_value = np.abs(mix_signal).max()
mix_normalized = mix_signal / peak_value if peak_value > 0 else mix_signal

assert np.isclose(np.abs(mix_normalized).max(), 1.0), "Normalization failed."

print(f"Shape of stems_stack: {stems_stack.shape}")
print(f"RMS Value: {rms_value:.6f}")
print(f"Max Amplitude (Peak): {peak_value:.6f}")


Shape of stems_stack: (4, 110250)
RMS Value: 0.102124
Max Amplitude (Peak): 0.589388
